In [ ]:

import base64, io, subprocess, sys, zipfile
from pathlib import Path

import importlib.metadata as metadata

PINS = ['qiskit==2.5.2', 'qiskit-aer==0.17.2']
EXTRA = []
BUNDLE = "..."

out = Path("/kaggle/working/output")
out.mkdir(parents=True, exist_ok=True)
# The image's numpy and scipy, and its JAX and libtpu where present, stay
# fixed while the pins install.
frozen = []
for name in ("numpy", "scipy", "jax", "jaxlib", "libtpu"):
    try:
        frozen.append(f"{name}=={metadata.version(name)}")
    except metadata.PackageNotFoundError:
        pass
Path("/tmp/constraints.txt").write_text("\n".join([*PINS, *frozen]) + "\n")
pip = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-c", "/tmp/constraints.txt", *PINS, *EXTRA],
    capture_output=True,
    text=True,
)
(out / "pip.txt").write_text(pip.stdout + pip.stderr)
if pip.returncode:
    raise SystemExit(f"pip install failed: {pip.returncode}")
for pin in [*PINS, *EXTRA, *frozen]:
    name, wanted = pin.split("==")
    if metadata.version(name) != wanted:
        raise SystemExit(f"{name} is {metadata.version(name)}, pinned {wanted}")
repo = Path("/tmp/gororoba/repo")
zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))).extractall(repo)
raise SystemExit(subprocess.run([sys.executable, "-B", "kaggle_kernel/kernel_main.py"], cwd=repo).returncode)
